## วิธีใช้  
### ทดลองใช้ฟังชั่นโดยแบ่งเป็นช่องๆ อันไหนดีก็ก้อปออกไปใช้ แต่ละช่องมัน เป็น independence แต่ก็สามารถต่อเนื่องได้ด้วย
---

In [27]:
def show_input_order(order):
    order_name = order
    print("Orderที่รับเข้ามา: ",order)
    return order_name
    

---

## Tricks หาที่อยู่ลูกค้า  
> แง่ดี
>> 1.ใน export file ของ shopee **_กรณีเป็นใบกำกับ_** จะ มี แขวง เขต จังหวัด ครบ  
>> 2.ใน export file ของ shopee **_ค่าจาก column อำเภอ จะเป็น แบบเต็ม_** เสมอ เช่น อำเภอธัญบุรี, เขตคลองสามวา

> แง่เลว
>> 1.ใน export file ของ shopee ค่าใน Column อาจจะเป็นภาษาอังกฤษ ทุก column  
>> 2.ใน Tambon_Data Columnตำบล ไม่มีคำว่าตำบล แต่ Columnอำเภอมีคำว่า อำเภอ แต่ เขต กับ แขวง มีทั้งคู่ สรุปถ้าใช้เพียวๆจะได้  "โพหัก อำเภอบางแพ ราชบุรี 70160"

In [59]:
import pandas as pd
import re

def clean_address(address):
    keywords = ["เขต", "แขวง", "ต.", "ตำบล", "อ.", "อำเภอ", "จ.", "จังหวัด"]

    # ตรวจสอบว่าสตริงมีคำ "จังหวัด" และ ("เขต" หรือ "แขวง") หรือไม่
    if "จังหวัด" in address and any(keyword in address for keyword in ["เขต", "แขวง"]):
        # ลบคำ "จังหวัด" ออกจากสตริง
        address = address.replace("จังหวัด", "")
        
    if "\n" in address:
        address = address.replace('\n', " ")

    # เริ่มต้นโดยการแยกคำด้วยช่องว่าง
    parts = address.split()
    
    # สร้าง list เพื่อเก็บคำที่ไม่ใช่คำย่อ
    cleaned_parts = []
    
    for part in parts:
        # ตรวจสอบว่าคำนี้เป็นคำย่อหรือไม่
        is_abbreviation = any(part.startswith(keyword) for keyword in ["ต.", "อ.", "จ."])
        
        if not is_abbreviation:
            cleaned_parts.append(part)
    
    # นำคำที่ไม่ใช่คำย่อมาเชื่อมกลับเป็นสตริงใหม่
    cleaned_address = ' '.join(cleaned_parts)
    
    # ลบคำที่มีส่วนที่เหมือนกันออก
    cleaned_address = clean_duplicate_parts(cleaned_address)
    
    # แก้ไขเครื่องหมายช่องว่างที่เหลือหลังการลบคำ
    cleaned_address = cleaned_address.replace("  ", " ")
    
    return cleaned_address

shopee_export_file = "../excel/Order.toship.20230903_20230914 (1).xlsx"
order = "230909SXC7FH38"
## ตำบล=แขวง// อำเภอ=เขต // จังหวัด 
def find_tambon():
    ##เตรียมข้อมูล Pattern ที่อยู่คนไทย
    shopee_data = "../excel/Order.toship.20230903_20230914 (1).xlsx"
    shopee_df = pd.read_excel(shopee_data)

    target_row_index = shopee_df.index[shopee_df.iloc[:,0] == order]
    
    cus_address = shopee_df.iloc[target_row_index, 15][0]
    print("cus_address", cus_address)
    amphoe = str(shopee_df.iloc[target_row_index, 18][0])
    print("amphoe", amphoe)
    postal_code = str(shopee_df.iloc[target_row_index, 20][0])
    
    ##เอาข้อมูลลูกค้ามาเทียบกับตาราง Pattern ที่อยู่คนไทย
    ##จัวนี้ต้องผูกกับ exe
    tambon_data_address = r"../excel/Addresscleaner_TambonData.xlsx"
    df_thai_add = pd.read_excel(tambon_data_address)
    allfiltered_df = df_thai_add[(df_thai_add['PostCodeMain'].astype(str) == postal_code) & (df_thai_add['DistrictThai'] == amphoe)]
    possible_tambon = list(allfiltered_df['TambonThai'])
    print("ไม่ได้เหรอ: ",possible_tambon)

    ##
    decent_tambon = []
    for tambon in possible_tambon:
        if tambon in cus_address:
            decent_tambon.append(tambon)
        else:
            pass

    return decent_tambon

find_tambon()


cus_address กรมสวัสดิการและคุ้มครองแรงงาน, ถนน มิตรไมตรี, แขวง ดินแดง  เขตดินแดง จังหวัดกรุงเทพมหานคร 10400
amphoe เขตดินแดง
ไม่ได้เหรอ:  ['แขวงดินแดง', 'แขวงรัชดาภิเษก']


[]

---